# TP2 : Apprentissage supervisé

## Informations

Étudiants
- Thibault Eynard-Suarez
- Enzo Marietti
- Adrien Cordero
Groupe : B1

### Description
Ce document fait office de rapport quant à notre analyse des jeux de données des TP 1 et 2 du cours d'analyse descriptive.

### I.A. générative
Nous avons utilisé l'IA générative pour :
- récupérer des plages de données depuis la doc ;
- fournir le code permettant d'afficher certains graphiques.

Nous n'avons pas utilisé l'IA pour le plan du rapport ou l'analyse, tous les textes en markdown ont été écrits à la main.

## Introduction

L'objectif de ce TP est d'entraîner des modèles à prédire si un individu à un revenu supérieur à 50 000\$. Nous allons utilisé le jeu de données ACS Income pour l'entrainement.

In [11]:
## Importation des librairies nécessaires

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

## Importation des jeux de données

# Colorado
df_co_ft = pd.read_csv("../../Dataset/Data/Colorado/acsincome_co_allfeatures.csv") # dataset n°0
df_co_lbl = pd.read_csv("../../Dataset/Data/Colorado/acsincome_co_label.csv")      # dataset n°1

# California
df_ca_ft = pd.read_csv("../../Dataset/Data/California/acsincome_ca_features.csv")  # dataset n°2
df_ca_lbl = pd.read_csv("../../Dataset/Data/California/acsincome_ca_labels.csv")   # dataset n°3

# Nebraska
df_ne_ft = pd.read_csv("../../Dataset/Data/Nebraska/acsincome_ne_allfeatures.csv") # dataset n°4
df_ne_lbl = pd.read_csv("../../Dataset/Data/Nebraska/acsincome_ne_label.csv")      # dataset n°5

datasets = [df_co_ft, df_co_lbl, df_ca_ft, df_ca_lbl, df_ne_ft, df_ne_lbl]

In [ ]:
import pickle 

safe = open("safe.pkl", "wb")

## Pré-traitement



- Retirer invidus resp et relations
- fusionner datasets des 3 états (en rajoutant un feature état)
- Données numériques -> normaliser (centrer + réduire)
- Donées catégoriques -> One hot encoding (rester sur catégories issues pour certains paramètres ?)
- Différences entre les états ?

In [12]:
census_occupation_subcategories_2018 = {
    # Sous-catégories de "Management, Business, Science, and Arts Occupations" (0010-3550)
    "Management, Business, and Financial Occupations": [10, 960],  #[cite: 81]
    "Computer, Engineering, and Science Occupations": [1005, 1980],  #[cite: 82]
    "Education, Legal, Community Service, Arts, and Media Occupations": [2001, 2970],  #[cite: 84]
    "Healthcare Practitioners and Technical Occupations": [3000, 3550],  #[cite: 86]
   
    # Sous-catégories de "Service Occupations" (3601-4655)
    "Healthcare Support Occupations": [3601, 3655],  #[cite: 87]
    "Protective Service Occupations": [3700, 3960],  #[cite: 87]
    "Food Preparation and Serving Related Occupations": [4000, 4160],  #[cite: 88]
    "Building and Grounds Cleaning and Maintenance Occupations": [4200, 4255],  #[cite: 88]
    "Personal Care and Service Occupations": [4330, 4655],  #[cite: 88]
   
    # Sous-catégories de "Sales and Office Occupations" (4700-5940)
    "Sales and Related Occupations": [4700, 4965],  #[cite: 89]
    "Office and Administrative Support Occupations": [5000, 5940],  #[cite: 89]
   
    # Sous-catégories de "Natural Resources, Construction, and Maintenance Occupations" (6005-7640)
    "Farming, Fishing, and Forestry Occupations": [6005, 6130],  #[cite: 90]
    "Construction and Extraction Occupations": [6200, 6950],  #[cite: 91]
    "Installation, Maintenance, and Repair Occupations": [7000, 7640],  #[cite: 92]
   
    # Sous-catégories de "Production, Transportation, and Material Moving Occupations" (7700-9760)
    "Production Occupations": [7700, 8990],  #[cite: 93]
    "Transportation and Material Moving Occupations": [9005, 9760],  #[cite: 94]
   
    # Catégorie militaire
    "Military Specific Occupations": [9800, 9830],  #[cite: 95]
}

ancestry_codes = {
    'WESTERN EUROPE (EXCEPT SPAIN)': [[1, 99], []],
    'EASTERN EUROPE AND FORMER SOVIET UNION': [[100, 180], [110, 121, 134, 135, 136, 137, 138, 139, 141, 149, 151, 162, 166]],
    'EUROPE, N.E.C.': [[181, 199], [182, 184, 186, 188, 189, 192, 197, 198, 199]],
    'HISPANIC CATEGORIES (INCLUDING SPAIN)': [[200, 299], [209, 216, 217, 220, 228, 230, 240, 241, 242, 243, 244, 245, 246, 247, 253, 254, 255, 256, 257, 258, 259, 260, 262, 263, 264, 265, 266, 267, 268, 269, 270, 272, 273, 274, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 296, 297, 298, 299]],
    'WEST INDIES (EXCEPT HISPANIC)': [[300, 359], [305, 306, 307, 309, 313, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359]],
    'CENTRAL AND SOUTH AMERICA (EXCEPT HISPANIC)': [[360, 399], [361, 362, 363, 364, 366, 367, 368, 369, 371, 372, 373, 374, 376, 377, 378, 379, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399]],
    'NORTH AFRICA AND SOUTH WEST ASIA': [[400, 499], [401, 403, 405, 409, 410, 418, 420, 424, 426, 428, 430, 432, 433, 440, 443, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 468, 469, 472, 473, 474, 475, 476, 477, 478, 479, 481, 482, 486, 487, 488, 489, 491, 492, 493, 494, 497, 498, 499]],
    'SUBSAHARAN AFRICA': [[500, 599], [501, 503, 505, 507, 509, 511, 514, 517, 518, 524, 526, 528, 533, 535, 536, 537, 539, 540, 542, 544, 548, 552, 559, 560, 562, 563, 565, 567, 575, 581, 585, 587]],
    'SOUTH ASIA': [[600, 699], [604, 605, 606, 608, 610, 611, 612, 613, 614, 617, 619, 621, 623, 625, 627, 629, 631, 633, 635, 637, 639, 641, 643, 645, 647, 649, 651, 653, 655, 657, 659, 660, 661, 662, 663, 664, 665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 676, 677, 678, 679, 681, 682, 683, 684, 685, 686, 687, 688, 689, 693, 694, 696, 697, 698, 699]],
    'OTHER ASIA': [[700, 799], [701, 705, 710, 711, 713, 715, 717, 719, 721, 722, 723, 724, 725, 726, 727, 728, 729, 731, 733, 735, 737, 738, 739, 747, 749, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 764, 767, 769, 772, 773, 775, 779, 780, 781, 784, 789, 791, 796, 797, 798, 799]],
    'PACIFIC': [[800, 899], [806, 807, 812, 835, 836, 837, 838, 839, 842, 848, 849, 851, 852, 853, 854, 855, 856, 857, 858, 859, 861, 863, 864, 865, 866, 867, 868, 869, 870, 871, 872, 873, 874, 875, 876, 877, 878, 879, 880, 881, 882, 883, 884, 885, 886, 887, 888, 889, 890, 891, 892, 893, 894, 895, 896, 897, 898, 899]],
    'NORTH AMERICA (EXCEPT HISPANIC)': [[900, 994], [909, 910, 911, 912, 915, 916, 926, 932, 938, 975, 992]],
    'RESIDUAL AND NO RESPONSE': [[995, 999], [997]],
}

nbCategories = {
    "nPOBP" : len(ancestry_codes.keys()),
    "nOCCP" :len(census_occupation_subcategories_2018.keys()),
    "COW" : 9,
    "SCHL" : 24,
    "MAR" : 5,
    "RELP" : 18,
    "SEX" : 2,
    "RAC1P" : 9,
    "PINCP" : 2
}

def categoriser_POBP(x):
    if pd.isna(x):
        return np.nan

    for i, (nom, (plage, exclusions)) in enumerate(ancestry_codes.items(), start=1):
        debut, fin = plage

        if debut <= x <= fin:
            if nom == "RESIDUAL AND NO RESPONSE":
                return np.nan
            return i

    return np.nan


def categoriser_OCCP(x):
    if pd.isna(x):
        return np.nan

    for i, (nom, (debut, fin)) in enumerate(
        census_occupation_subcategories_2018.items(), start=1
    ):
        if debut <= x <= fin:
            return i

    return np.nan


for i in [0, 2, 4]:
    datasets[i]["nPOBP"] = datasets[i]["POBP"].apply(categoriser_POBP)
    datasets[i]["nOCCP"] = datasets[i]["OCCP"].apply(categoriser_OCCP)

In [13]:
datasets[0]["State"] = [[1, 0, 0] for i in range(len(datasets[0]))]
datasets[1]["State"] = [[1, 0, 0] for i in range(len(datasets[0]))]

datasets[2]["State"] = [[0, 1, 0] for i in range(len(datasets[2]))]
datasets[3]["State"] = [[0, 1, 0] for i in range(len(datasets[2]))]

datasets[4]["State"] = [[0, 0, 1] for i in range(len(datasets[4]))]
datasets[5]["State"] = [[0, 0, 1] for i in range(len(datasets[4]))]

datasets

[       AGEP  COW  SCHL  MAR    OCCP   POBP  RELP  WKHP  SEX  RAC1P  nPOBP  \
 0      19.0  2.0  19.0  5.0  2850.0    6.0  17.0   2.0  2.0    9.0      1   
 1      24.0  5.0  18.0  5.0  9645.0   13.0  17.0  40.0  1.0    2.0      1   
 2      22.0  1.0  19.0  5.0  1520.0   53.0  17.0  45.0  2.0    1.0      1   
 3      32.0  1.0  16.0  1.0  4055.0    8.0  16.0  40.0  2.0    1.0      1   
 4      19.0  2.0  19.0  5.0  2006.0   34.0  17.0   9.0  2.0    1.0      1   
 ...     ...  ...   ...  ...     ...    ...   ...   ...  ...    ...    ...   
 31301  52.0  1.0  21.0  1.0  5000.0   35.0   0.0  40.0  2.0    1.0      1   
 31302  54.0  1.0  19.0  1.0   910.0   35.0   1.0  40.0  1.0    1.0      1   
 31303  39.0  1.0  22.0  1.0  1021.0  207.0   0.0  40.0  1.0    6.0      4   
 31304  39.0  1.0  24.0  1.0  2205.0  207.0   1.0  40.0  2.0    6.0      4   
 31305  55.0  3.0  20.0  3.0  3820.0   38.0   0.0  40.0  1.0    1.0      1   
 
        nOCCP      State  
 0          3  [1, 0, 0]  
 1      

In [14]:
datasets[0]["Label"] = [datasets[1]["PINCP"][i] for i in range(len(datasets[0]))]

datasets[2]["Label"] = [datasets[3]["PINCP"][i] for i in range(len(datasets[2]))]

datasets[4]["Label"] = [datasets[5]["PINCP"][i] for i in range(len(datasets[4]))]


In [15]:
data = pd.concat([datasets[0], datasets[2], datasets[4]])
data

,AGEP,COW,SCHL,MAR,OCCP,POBP,RELP,WKHP,SEX,RAC1P,nPOBP,nOCCP,State,Label
0,19.0,2.0,19.0,5.0,2850.0,6.0,17.0,2.0,2.0,9.0,1,3,"[1, 0, 0]",0
1,24.0,5.0,18.0,5.0,9645.0,13.0,17.0,40.0,1.0,2.0,1,16,"[1, 0, 0]",0
2,22.0,1.0,19.0,5.0,1520.0,53.0,17.0,45.0,2.0,1.0,1,2,"[1, 0, 0]",0
3,32.0,1.0,16.0,1.0,4055.0,8.0,16.0,40.0,2.0,1.0,1,7,"[1, 0, 0]",0
4,19.0,2.0,19.0,5.0,2006.0,34.0,17.0,9.0,2.0,1.0,1,3,"[1, 0, 0]",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10780,59.0,1.0,20.0,5.0,1430.0,20.0,0.0,50.0,1.0,1.0,1,2,"[0, 0, 1]",1
10781,59.0,3.0,22.0,3.0,1360.0,8.0,0.0,45.0,2.0,1.0,1,2,"[0, 0, 1]",1
10782,62.0,1.0,19.0,3.0,20.0,19.0,0.0,50.0,2.0,1.0,1,1,"[0, 0, 1]",1
10783,47.0,1.0,16.0,5.0,6050.0,31.0,0.0,65.0,1.0,1.0,1,12,"[0, 0, 1]",0


On supprime les anciennes colonnes OCCP et POBP ainsi que la colonne RELP car, au vu de l'analyse que nous avons précédemment effectué, elle nous paraît avoir peu d'incidence sur la prédiction.

In [ ]:
data = data.drop(columns=["OCCP", "POBP", "RELP"])

,AGEP,COW,SCHL,MAR,WKHP,SEX,RAC1P,nPOBP,nOCCP,State,Label
0,19.0,2.0,19.0,5.0,2.0,2.0,9.0,1,3,"[1, 0, 0]",0
1,24.0,5.0,18.0,5.0,40.0,1.0,2.0,1,16,"[1, 0, 0]",0
2,22.0,1.0,19.0,5.0,45.0,2.0,1.0,1,2,"[1, 0, 0]",0
3,32.0,1.0,16.0,1.0,40.0,2.0,1.0,1,7,"[1, 0, 0]",0
4,19.0,2.0,19.0,5.0,9.0,2.0,1.0,1,3,"[1, 0, 0]",0
...,...,...,...,...,...,...,...,...,...,...,...
10780,59.0,1.0,20.0,5.0,50.0,1.0,1.0,1,2,"[0, 0, 1]",1
10781,59.0,3.0,22.0,3.0,45.0,2.0,1.0,1,2,"[0, 0, 1]",1
10782,62.0,1.0,19.0,3.0,50.0,2.0,1.0,1,1,"[0, 0, 1]",1
10783,47.0,1.0,16.0,5.0,65.0,1.0,1.0,1,12,"[0, 0, 1]",0


In [ ]:
def center_reduct(x, moy, std):
    return (x - moy) / std

def center_reduct_column(name):
    moy = data[name].mean()
    std = data[name].std()
    data[name] = data[name].apply(lambda x : center_reduct(x, moy, std))

center_reduct_column("AGEP")
center_reduct_column("WKHP")

,AGEP,COW,SCHL,MAR,WKHP,SEX,RAC1P,nPOBP,nOCCP,State,Label
0,-1.589512,2.0,19.0,5.0,-2.748587,2.0,9.0,1,3,"[1, 0, 0]",0
1,-1.255619,5.0,18.0,5.0,0.147017,1.0,2.0,1,16,"[1, 0, 0]",0
2,-1.389176,1.0,19.0,5.0,0.528018,2.0,1.0,1,2,"[1, 0, 0]",0
3,-0.721390,1.0,16.0,1.0,0.147017,2.0,1.0,1,7,"[1, 0, 0]",0
4,-1.589512,2.0,19.0,5.0,-2.215186,2.0,1.0,1,3,"[1, 0, 0]",0
...,...,...,...,...,...,...,...,...,...,...,...
10780,1.081633,1.0,20.0,5.0,0.909018,1.0,1.0,1,2,"[0, 0, 1]",1
10781,1.081633,3.0,22.0,3.0,0.528018,2.0,1.0,1,2,"[0, 0, 1]",1
10782,1.281969,1.0,19.0,3.0,0.909018,2.0,1.0,1,1,"[0, 0, 1]",1
10783,0.280290,1.0,16.0,5.0,2.052020,1.0,1.0,1,12,"[0, 0, 1]",0


In [18]:
def one_hot_encoding(column, x):
    n = nbCategories[column]
    ls = [0 for i in range(n)]
    ls[int(x)-1] = 1
    return ls

def one_hot_encoding_apply(column):
    data[column] = data[column].apply(lambda x : one_hot_encoding(column, x))

one_hot_encoding_apply("COW")
one_hot_encoding_apply("SCHL")
one_hot_encoding_apply("MAR")
one_hot_encoding_apply("SEX")
one_hot_encoding_apply("RAC1P")
one_hot_encoding_apply("nPOBP")
one_hot_encoding_apply("nOCCP")

Voici a quoi ressemblent les données à la fin du pré-traitement

In [19]:
data

,AGEP,COW,SCHL,MAR,WKHP,SEX,RAC1P,nPOBP,nOCCP,State,Label
0,-1.589512,"[0, 1, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1]",-2.748587,"[0, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 1]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0]",0
1,-1.255619,"[0, 0, 0, 0, 1, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1]",0.147017,"[1, 0]","[0, 1, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0]",0
2,-1.389176,"[1, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1]",0.528018,"[0, 1]","[1, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0]",0
3,-0.721390,"[1, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0]",0.147017,"[0, 1]","[1, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0]",0
4,-1.589512,"[0, 1, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1]",-2.215186,"[0, 1]","[1, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0]",0
...,...,...,...,...,...,...,...,...,...,...,...
10780,1.081633,"[1, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1]",0.909018,"[1, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 1]",1
10781,1.081633,"[0, 0, 1, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 1, 0, 0]",0.528018,"[0, 1]","[1, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 1]",1
10782,1.281969,"[1, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 1, 0, 0]",0.909018,"[0, 1]","[1, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 1]",1
10783,0.280290,"[1, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1]",2.052020,"[1, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...","[0, 0, 1]",0


In [ ]:
pickle.dump(data, safe)